In [ ]:
import mne
import pandas as pd

In [ ]:
hd = '/Volumes/evan_ext/EEG Analysis/'
mne.set_log_level('ERROR')

In [ ]:
def epoch_raw_data(file_path,max_freq=45, downsample_freq=250):
    """
    Load EEG, pick key channels + stim, filter, (optionally) downsample, and epoch around events.
    For .mff, the stim channel may be 'DIN4' or '40hz'—we detect it automatically.
    """
    file_path = str(file_path)  # in case a Path is passed
    # --- montage ---
    montage = mne.channels.read_custom_montage(hd + 'data/9_18AverageNet128_v1.sfp')

    # channels needed for analysis (EEG only; stim added dynamically)
    eeg_channels = ['E106', 'E7', 'E57', 'E101']

    # keep logs quiet
    mne.set_log_level('ERROR')

    if file_path.lower().endswith('.mff'):
        # Load
        raw = mne.io.read_raw_egi(
            file_path,
            eog=None, misc=None,
            preload=True,
            events_as_annotations=False
        )

        # --- determine stim channel (DIN4 preferred, else 40hz) ---
        chs = set(raw.ch_names)
        if 'DIN4' in chs:
            stim_ch = 'DIN4'
        elif '40hz' in chs:
            stim_ch = '40hz'
        else:
            raise RuntimeError(
                "No stim channel found. Expected 'DIN4' or '40hz' in the .mff file."
            )

        # --- pick available EEG + stim ---
        picks_to_keep = [ch for ch in eeg_channels if ch in chs] + [stim_ch]
        if stim_ch not in picks_to_keep:
            raise RuntimeError(f"Stim channel {stim_ch} not present after picking.")
        raw.pick(picks_to_keep, exclude=[])

        # --- filtering (EEG only) ---
        raw.filter(l_freq=1, h_freq=max_freq, picks='eeg')

        # --- set reference & montage ---
        # Only use refs that actually exist after picking
        ref_available = [ch for ch in ['E57', 'E101'] if ch in raw.ch_names]
        if len(ref_available) == 2:
            raw.set_eeg_reference(ref_channels=ref_available)  # type: ignore
        else:
            # fall back to average if either ref missing
            raw.set_eeg_reference('average')  # type: ignore

        raw.set_montage(montage)

        # --- (optional) downsample BEFORE event finding ---
        if downsample_freq:
            raw.resample(downsample_freq)

        # --- events & epochs ---
        events = mne.find_events(raw, stim_channel=stim_ch)
        epochs = mne.Epochs(
            raw,
            events,
            tmin=-0.5,
            tmax=1.0,
            baseline=(-0.1, 0),
            reject=dict(eeg=120e-6),
            detrend=1,
            preload=True,
        )

        # drop refs (if there) and the stim channel we used
        to_drop = [ch for ch in ['E57', 'E101', stim_ch] if ch in epochs.ch_names]
        if to_drop:
            epochs.drop_channels(to_drop)

        return epochs


    raise ValueError("Unsupported file type. Expected a .mff or .set file.")

In [ ]:
from pathlib import Path

def exclude_flag(group, n, had_err=False):
    if had_err: return 1
    g = str(group).upper()
    if g in {"ASD", "PMS"}:  return 1 if (n or 0) < 25 else 0
    if g == "TD":            return 1 if (n or 0) < 40 else 0
    return 1 if (n or 0) <= 0 else 0

BASE = Path(f"/Volumes/evan_ext/EEG Analysis/analyses/Epoched Data/Final Epochs (max freq 45Hz)")

df = pd.read_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv")
df = df.rename(columns=lambda c: c.strip().strip("'").strip('"'))

for i, row in df.iterrows():
    pid   = str(row["Participant ID"]).strip()
    site  = str(row["Site"]).strip()
    group = str(row["Group"]).strip()
    mff   = str(row["MFF Path"]).strip()
    cur   = str(row.get("Epoch Files", "") or "").strip()


    out = BASE / site / group / f"{pid}_epoched_data.fif"
    out.parent.mkdir(parents=True, exist_ok=True)

    # --- check existing files ---
    resolved = None
    if cur and Path(cur).is_file():
        resolved = Path(cur)
        print(f"[SKIP] {pid}: existing Epoch Files path found ({cur})")
    elif out.is_file():
        resolved = out
        print(f"[SKIP] {pid}: canonical epoched file already exists ({out})")

    if resolved:
        df.at[i, "Epoch Files"] = str(resolved)
        continue

    # --- epoching ---
    print(f"[EPOCH] {pid}: starting epoching (Site={site}, Group={group})")
    n = None
    had_err = False
    try:
        mff_path = Path(mff)

        ep = epoch_raw_data(str(mff_path))
        n = int(len(ep))
        ep.save(str(out), overwrite=True)

        df.at[i, "Epoch Files"] = str(out)
        df.at[i, "Number of Samples"] = float(n)
        print(f"[DONE] {pid}: epoched successfully with {n} samples → {out}")
    except Exception as e:
        had_err = True
        print(f"[FAIL] {pid}: {type(e).__name__}: {e}")

    df.at[i, "Exclude"] = exclude_flag(group, n, had_err)

df.to_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv", index=False)
print("CSV updated.")

In [ ]:
df = pd.read_csv("/Volumes/evan_ext/EEG Analysis/data/participant_info.csv")
df = df.rename(columns=lambda c: c.strip().strip("'").strip('"'))
df = df[df['Site']== 'MSH']
df = df[df['Exclude']==0]

df = df[~df['Participant ID'].isin(['FF0261', 'FF0661'])] #hand remove subjects with problamatic waveforms due to high 40 hz power likely caused by audio leakage

study_xlsx = "/Volumes/evan_ext/EEG Analysis/data/StudyIDs_JS.xlsx"
study_df = pd.read_excel(study_xlsx,header=0)

df = df.merge(
    study_df[['FFID', 'Age_years','Sex','IQ_score','ADOS_comp_score']],     # only keep the needed columns
    how='left',                     # keep all rows from df, fill missing with NaN
    left_on='Participant ID',       # match key from df
    right_on='FFID'                 # match key from ff
)

df['IQ_score'] = pd.to_numeric(df['IQ_score'], errors='coerce')
df['ADOS_comp_score'] = pd.to_numeric(df['ADOS_comp_score'], errors='coerce')

# optionally drop the redundant 'FFID' column now that it's merged
df = df.drop(columns='FFID')
df = df[~df['Age_years'].isna()]
df

In [ ]:
# --- numeric summaries per Group ---
numeric_summary = (
    df.groupby('Group', as_index=False)
      .agg(
          n=('Participant ID', 'size'),
          Age_years_mean=('Age_years', 'mean'),
          Age_years_sd=('Age_years', 'std'),
          Age_years_min=('Age_years', 'min'),
          Age_years_max=('Age_years', 'max'),
          IQ_score_mean=('IQ_score', 'mean'),
          IQ_score_sd=('IQ_score', 'std'),
          IQ_score_count=('IQ_score', 'count'),
          ADOS_comp_mean=('ADOS_comp_score','mean'),
          ADOS_comp_sd=('ADOS_comp_score','std'),
          ADOS_comp_count=('ADOS_comp_score','count'),
          num_samples_mean=('Number of Samples','mean'),
          num_samples_std=('Number of Samples','std'),
      )
)

# --- sex distribution (counts + %) per Group ---
sex_summary = (
    df.groupby(['Group', 'Sex'], dropna=False)
      .size()
      .reset_index(name='n')
)

# aligned percent using transform (prevents the ValueError)
sex_summary['percent'] = (
    sex_summary['n'] /
    sex_summary.groupby('Group')['n'].transform('sum') * 100
).round(1)

# (optional) pivot Sex to columns so you can join back to numeric_summary
sex_counts_wide = (
    sex_summary
      .pivot(index='Group', columns='Sex', values='n')
      .fillna(0)
      .astype(int)
      .reset_index()
)
sex_perc_wide = (
    sex_summary
      .pivot(index='Group', columns='Sex', values='percent')
      .fillna(0)
      .reset_index()
      .add_prefix('pct_')  # columns like pct_Female, pct_Male
      .rename(columns={'pct_Group': 'Group'})
)

# combine (optional)
summary_combined = (
    numeric_summary
      .merge(sex_counts_wide, on='Group', how='left')
      .merge(sex_perc_wide, on='Group', how='left')
)

# ------------------------------------------------------------
# Format summary table for copy/paste in the requested style
# ------------------------------------------------------------

output_rows = []

for _, row in summary_combined.iterrows():
    group = row["Group"]

    # --- n ---
    n = int(row["n"])

    # --- Female counts & percent ---
    # assumes sex columns = "F" and "M" in sex_counts_wide
    female_n = int(row.get("F", 0))
    female_pct = row.get("pct_F", 0.0)
    female_str = f"{female_n} ({female_pct:.1f}%)"

    # --- Age ---
    age_mean = row["Age_years_mean"]
    age_sd   = row["Age_years_sd"]
    age_min  = row["Age_years_min"]
    age_max  = row["Age_years_max"]
    age_str = f"{age_mean:.1f} ± {age_sd:.1f} [{age_min:.0f}, {age_max:.0f}]"

    # --- IQ ---
    iq_mean = row["IQ_score_mean"]
    iq_sd   = row["IQ_score_sd"]

    if pd.isna(iq_mean):
        iq_str = "NA"
    else:
        iq_str = f"{iq_mean:.1f} ± {iq_sd:.1f}"

    # --- ADOS comp ---
    adosc_mean = row["ADOS_comp_mean"]
    adosc_sd   = row["ADOS_comp_sd"]
    
    if pd.isna(adosc_mean):
        adosc_str = "NA"
    else:
        adosc_str = f"{adosc_mean:.1f} ± {adosc_sd:.1f}"


    # Store row
    output_rows.append([
        group,
        str(n),
        female_str,
        age_str,
        iq_str,
        adosc_str,
    ])

# ------------------------------------------------------------
# Print final table for copy/paste
# ------------------------------------------------------------
print("Group | n | Female (%) | Age (years) (mean ± SD) [min, max] | IQ (mean ± SD) | ADOS Comp (mean ± SD] | ADOS Total (mean ± SD]")
print("-" * 110)
for r in output_rows:
    print(" | ".join(r))


In [ ]:
summary_combined.iloc[:,3:]

In [ ]:
import numpy as np
from neurodsp.spectral import compute_spectrum_welch
from fooof import FOOOF

# =========================
# --- Helper Functions ----
# =========================
def get_ts(epoch: mne.Epochs):
    """Compute mean time series across channels and trials."""
    return np.mean(epoch.get_data(), axis=(0, 1))

def get_power_spectrum(ts):
    """Compute power spectrum from time series."""
    freqs, power = compute_spectrum_welch(ts,250)
    return freqs, power

def get_fooof_params(freqs, power):
    """Fit FOOOF model and return paramaters and goodness of fit metrics"""
    fm = FOOOF(verbose=False)
    freq_range = [2, 50]
    fm.fit(freqs,power,freq_range)
    results_dict = {'n_peaks' : fm.n_peaks_,
                    'r_squared' : fm.r_squared_,
                    'error' : fm.error_,
                    'aperiodic_offset' : fm.aperiodic_params_[0],
                    'aperiodic_exponent' : fm.aperiodic_params_[1],
                    'peak_centers' : fm.peak_params_[:,0],
                    'peak_powers' : fm.peak_params_[:,1],
                    'peak_bandwidths' : fm.peak_params_[:,2]
    }
    return results_dict

BANDS = [
    ("theta", (4, 7)),
    ("alpha", (8, 12)),
    ("beta", (13, 30)),
    ("gamma", (31, 45)),
]

def featurize_peaks_by_band(peak_params, bands=BANDS):
    """
    Convert raw FOOOF peak parameters (n_peaks x 3) into engineered
    band-wise features.

    Returns:
      feats: 1D numpy vector of all features
      names: list of feature names in order
    """
    if peak_params is None or len(peak_params) == 0:
        names, feats = [], []
        for bname, _ in bands:
            names += [
                f"{bname}_present", f"{bname}_count", f"{bname}_power_sum",
                f"{bname}_power_max", f"{bname}_center_max", f"{bname}_bw_mean",
                f"{bname}_center_mean"
            ]
            feats += [0, 0, 0.0, 0.0, 0.0, 0.0, 0.0]
        return np.asarray(feats, float), names

    c, p, w = peak_params[:,0], peak_params[:,1], peak_params[:,2]

    names, feats = [], []

    for bname, (lo, hi) in bands:
        mask = (c >= lo) & (c <= hi)
        count = int(mask.sum())
        present = int(count > 0)

        if count:
            c_band = c[mask]
            p_band = p[mask]
            w_band = w[mask]

            idx_max = np.argmax(p_band)

            power_sum   = float(p_band.sum())
            power_max   = float(p_band[idx_max])
            center_max  = float(c_band[idx_max])
            bw_mean     = float(w_band.mean())
            center_mean = float(c_band.mean())
        else:
            power_sum = power_max = center_max = bw_mean = center_mean = 0.0

        names += [
            f"{bname}_present", f"{bname}_count", f"{bname}_power_sum",
            f"{bname}_power_max", f"{bname}_center_max",
            f"{bname}_bw_mean", f"{bname}_center_mean"
        ]
        feats += [present, count, power_sum, power_max, center_max,
                  bw_mean, center_mean]

    return np.asarray(feats, float), names

def get_fooof_features(params):
    """
    Convert raw FOOOF param dictionary into:
        - Aperiodic features
        - Band-summaries of peaks using featurize_peaks_by_band()

    Returns:
        feature_vector: 1D numpy array
        feature_names: list of names
    """
    # ---- global FOOOF scalar features ----
    aperiodic_feats = np.array([
        params["r_squared"],
        params["error"],
        params["aperiodic_offset"],
        params["aperiodic_exponent"]
    ], float)
    aperiodic_names = [
        "r_squared",
        "error",
        "aperiodic_offset",
        "aperiodic_exponent"
    ]

    # ---- bandwise peak features ----
    peak_params = np.column_stack([
        params["peak_centers"],
        params["peak_powers"],
        params["peak_bandwidths"]
    ]) if len(params["peak_centers"]) else None

    band_feats, band_names = featurize_peaks_by_band(peak_params)

    # ---- combine ----
    feats = np.concatenate([aperiodic_feats, band_feats])
    names = aperiodic_names + band_names

    return feats, names

def get_itpc_features(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute per-channel ITPC, average across channels manually."""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0
    decim = 2

    itpc_list = []

    for ch_idx in range(len(epoch.ch_names)):
        _, itpc = mne.time_frequency.tfr_morlet(
            epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
            return_itc=True, decim=decim, average=True, picks=[ch_idx]
        )
        itpc_list.append(itpc.data.squeeze())

    itpc_mean = np.mean(itpc_list, axis=0)
    
    return itpc_mean

def get_ersp_features(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute ERSP over all channels and trials"""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0

    decim = 2
    baseline = (-.1, 0)
    baseline_mode = "logratio"

    power = mne.time_frequency.tfr_morlet(
        epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
        return_itc=False, decim=decim, average=True
    )

    power.apply_baseline(baseline=baseline, mode=baseline_mode)
    power_features = power.data.squeeze().mean(axis=0)

    return power_features

def collate_ASSR_features(group:str):
    sub_df = df[df['Group'] == group]
    group_ts = []
    group_itpc = []
    group_ersp = []

    for epoch_path in sub_df['Epoch Files']:
        epoch = mne.read_epochs(epoch_path, preload=True)
        ts = get_ts(epoch)
        group_ts.append(ts)

        # Get ITPC for each subject
        itpc = get_itpc_features(epoch, band_center=40, band_width=10)
        group_itpc.append(itpc)

        ersp = get_ersp_features(epoch, band_center=40, band_width=10)
        group_ersp.append(ersp)
    
    return (group_ts, group_itpc, group_ersp)

def collate_FOOOF_features(group: str):
    """
    Returns:
        X: (n_subjects, n_features) numpy array
        names: list of feature names
        avg_ts: average time series across subjects
    """

    sub_df = df[df['Group'] == group]

    feature_rows = []
    group_ts = []

    for epoch_path in sub_df['Epoch Files']:
        epoch = mne.read_epochs(epoch_path, preload=True)
        ts = get_ts(epoch)
        group_ts.append(ts)

        freqs, power = get_power_spectrum(ts)
        params = get_fooof_params(freqs, power)
        feats, names = get_fooof_features(params)

        feature_rows.append(feats)

    X = np.vstack(feature_rows)
    avg_ts = np.mean(group_ts, axis=0)

    return X, names, avg_ts


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_assr_features(groups=['TD', 'ASD', 'PMS']):
    """
    Plot ASSR time-series, ITPC, and ERSP for each group.
    """

    # ---------------------------
    # 1. Collect all features
    # ---------------------------
    results = {}

    for group in groups:
        ts_list, itpc_list, ersp_list = collate_ASSR_features(group)

        results[group] = {
            "TS": np.array(ts_list),
            "ITPC": np.array(itpc_list),
            "ERSP": np.array(ersp_list),
        }

    # Axes from ITPC/ERSP
    n_freqs = results[groups[0]]["ITPC"].shape[0]
    n_times = results[groups[0]]["ITPC"].shape[1]

    times = np.linspace(-.5, 1.0, n_times)
    freqs = np.linspace(35, 45, n_freqs)

    # ---------------------------------------------------
    # 2. Plot Time Series — consistent y-axis across groups
    # ---------------------------------------------------
    # Find global Y limits
    all_ts = np.concatenate([v["TS"] for v in results.values()], axis=0)
    global_ymin = np.min(all_ts)
    global_ymax = np.max(all_ts)

    fig, axes = plt.subplots(1, len(groups), figsize=(16, 4), constrained_layout=True)
    for ax, group in zip(axes, groups):

        t = np.linspace(-.5, 1.0, results[group]["TS"].shape[1])
        X = results[group]['TS']
        if X is not None:
            ax.plot(t, X.T, alpha=0.2, linewidth=1)
            ax.plot(t, X.mean(axis=0), linewidth=2.5, color="C0")

        ax.set_title(f"Time Series — {group}", fontsize=16, pad=10)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Amplitude")
        ax.axvline(0, color="k", linestyle="--", linewidth=1)

        # Apply consistent limits
        ax.set_ylim(global_ymin, global_ymax)

    plt.tight_layout()
    plt.savefig("figures/1a.svg", format="svg", bbox_inches='tight')
    plt.show()

    # --- 2. Global limits for ITPC / ERSP ---
    all_itpc = np.concatenate(
        [np.mean(np.stack(v["ITPC"]), axis=0).ravel() for v in results.values()]
    )
    itpc_vmin, itpc_vmax = np.percentile(all_itpc, [2, 98])

    all_power = np.concatenate(
        [np.mean(np.stack(v["ERSP"]), axis=0).ravel() for v in results.values()]
    )
    ersp_vmin, ersp_vmax = np.percentile(all_power, [2, 98])

    # ---------------------------------------------------
    # 3. Plot ITPC
    # ---------------------------------------------------
    fig, axes = plt.subplots(1, len(groups), figsize=(16, 4), constrained_layout=True)

    for ax, group in zip(axes, groups):
        mean_itpc = results[group]["ITPC"].mean(axis=0)

        im = ax.imshow(
            mean_itpc,
            aspect="auto",
            origin="lower",
            extent=[times[0], times[-1], freqs[0], freqs[-1]],
            cmap="plasma",
            vmin=itpc_vmin,
            vmax=itpc_vmax
        )

        ax.set_title(f"ITPC — {group}", fontsize=14, pad=10)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Frequency (Hz)")

    fig.colorbar(im, ax=axes, label="ITPC", fraction=0.035, pad=0.04)
    plt.savefig("figures/1b.svg", format="svg", bbox_inches='tight')
    plt.show()

    # ---------------------------------------------------
    # 4. Plot ERSP
    # ---------------------------------------------------
    fig, axes = plt.subplots(1, len(groups), figsize=(16, 4), constrained_layout=True)

    for ax, group in zip(axes, groups):
        mean_ersp = results[group]["ERSP"].mean(axis=0)

        im = ax.imshow(
            mean_ersp,
            aspect="auto",
            origin="lower",
            extent=[times[0], times[-1], freqs[0], freqs[-1]],
            cmap="viridis",
            vmin=ersp_vmin,
            vmax=ersp_vmax
        )

        ax.set_title(f"ERSP — {group}", fontsize=14, pad=10)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Frequency (Hz)")

    fig.colorbar(im, ax=axes, label="ERSP (baseline-corrected)", fraction=0.035, pad=0.04)
    plt.savefig("figures/1c.svg", format="svg", bbox_inches='tight')
    plt.show()

In [ ]:
plot_assr_features()

In [ ]:
import numpy as np
from scipy import stats
from itertools import combinations


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def cohens_d(a, b):
    """Pooled-SD Cohen's d (signed: positive = a > b)."""
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(
        ((na - 1) * np.std(a, ddof=1) ** 2 + (nb - 1) * np.std(b, ddof=1) ** 2)
        / (na + nb - 2)
    )
    return (np.mean(a) - np.mean(b)) / (pooled_sd + 1e-12)


def cohens_d_map(A, B):
    """
    Element-wise Cohen's d between two arrays of shape (n_subjects, n_freqs, n_times).
    Returns shape (n_freqs, n_times).
    """
    na, nb = A.shape[0], B.shape[0]
    pooled_var = (
        (na - 1) * np.var(A, axis=0, ddof=1) +
        (nb - 1) * np.var(B, axis=0, ddof=1)
    ) / (na + nb - 2)
    return (A.mean(axis=0) - B.mean(axis=0)) / (np.sqrt(pooled_var) + 1e-12)


def tstat_map(A, B):
    """Independent-samples t-statistic map, shape (n_freqs, n_times)."""
    na, nb = A.shape[0], B.shape[0]
    mean_diff = A.mean(axis=0) - B.mean(axis=0)
    pooled_se = np.sqrt(
        np.var(A, axis=0, ddof=1) / na +
        np.var(B, axis=0, ddof=1) / nb
    )
    return mean_diff / (pooled_se + 1e-12)


# ─────────────────────────────────────────────
# Cluster helpers
# ─────────────────────────────────────────────

def _threshold_tmap(tmap, df, alpha=0.05):
    """Return binary mask of supra-threshold pixels (two-tailed)."""
    t_thresh = stats.t.ppf(1 - alpha / 2, df)
    return np.abs(tmap) > t_thresh


def _label_clusters(mask):
    """
    Simple 4-connected flood-fill labelling on a 2-D binary mask.
    Returns (labels, n_clusters).
    """
    from collections import deque
    labels = np.zeros_like(mask, dtype=int)
    nf, nt = mask.shape
    current = 0
    for i0 in range(nf):
        for j0 in range(nt):
            if mask[i0, j0] and labels[i0, j0] == 0:
                current += 1
                queue = deque([(i0, j0)])
                labels[i0, j0] = current
                while queue:
                    i, j = queue.popleft()
                    for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                        ni, nj = i+di, j+dj
                        if 0 <= ni < nf and 0 <= nj < nt:
                            if mask[ni, nj] and labels[ni, nj] == 0:
                                labels[ni, nj] = current
                                queue.append((ni, nj))
    return labels, current


def _cluster_mass(tmap, labels, n_clusters):
    """Sum of t-values within each cluster (cluster mass statistic)."""
    masses = []
    for c in range(1, n_clusters + 1):
        masses.append(np.sum(tmap[labels == c]))
    return masses


def cluster_permutation_test(A, B, n_perm=1000, alpha=0.05, seed=42):
    """
    Non-parametric cluster-based permutation test for 2-D data
    (n_subjects × n_freqs × n_times).

    Returns
    -------
    dict with keys:
        observed_clusters   : list of dicts, one per observed cluster
            - 'mass'        : observed cluster mass (sum of t-values)
            - 'size'        : number of pixels
            - 'p_value'     : cluster-level p-value
            - 'sign'        : '+' or '-'
        max_cohen_d         : peak |Cohen's d| across the significant cluster(s)
        mean_cohen_d_sig    : mean |Cohen's d| within significant pixels
        n_sig_clusters      : number of clusters with p < alpha
        n_perm              : permutations run
        alpha               : threshold used
    """
    rng = np.random.default_rng(seed)
    na, nb = A.shape[0], B.shape[0]
    df = na + nb - 2
    combined = np.concatenate([A, B], axis=0)

    # Observed t-map and clusters
    obs_tmap = tstat_map(A, B)
    obs_mask = _threshold_tmap(obs_tmap, df, alpha)
    obs_labels, obs_n = _label_clusters(obs_mask)
    obs_masses = _cluster_mass(obs_tmap, obs_labels, obs_n)

    if obs_n == 0:
        return {
            "observed_clusters": [],
            "max_cohen_d": 0.0,
            "mean_cohen_d_sig": 0.0,
            "n_sig_clusters": 0,
            "n_perm": n_perm,
            "alpha": alpha,
        }

    # Permutation distribution of max cluster mass
    perm_max_pos = np.zeros(n_perm)
    perm_max_neg = np.zeros(n_perm)

    for p in range(n_perm):
        idx = rng.permutation(na + nb)
        pA = combined[idx[:na]]
        pB = combined[idx[na:]]
        pt = tstat_map(pA, pB)
        pmask = _threshold_tmap(pt, df, alpha)
        pl, pn = _label_clusters(pmask)
        if pn > 0:
            pm = _cluster_mass(pt, pl, pn)
            pos = [m for m in pm if m > 0]
            neg = [m for m in pm if m < 0]
            perm_max_pos[p] = max(pos) if pos else 0.0
            perm_max_neg[p] = min(neg) if neg else 0.0
        # else both stay 0

    # Cohen's d map
    d_map = cohens_d_map(A, B)

    cluster_results = []
    for c, mass in enumerate(obs_masses, start=1):
        c_mask = obs_labels == c
        if mass > 0:
            p_val = np.mean(perm_max_pos >= mass)
        else:
            p_val = np.mean(perm_max_neg <= mass)

        d_vals = np.abs(d_map[c_mask])
        cluster_results.append({
            "cluster_id": c,
            "mass": float(mass),
            "size": int(c_mask.sum()),
            "p_value": float(p_val),
            "sign": "+" if mass > 0 else "-",
            "peak_cohen_d": float(d_vals.max()),
            "mean_cohen_d": float(d_vals.mean()),
        })

    # Summary stats across ALL significant clusters
    sig = [r for r in cluster_results if r["p_value"] < alpha]
    if sig:
        sig_mask = np.zeros_like(obs_mask)
        for r in sig:
            sig_mask |= (obs_labels == r["cluster_id"])
        sig_d = np.abs(d_map[sig_mask])
        max_d_overall = float(sig_d.max())
        mean_d_overall = float(sig_d.mean())
    else:
        max_d_overall = 0.0
        mean_d_overall = 0.0

    return {
        "observed_clusters": cluster_results,
        "max_cohen_d": max_d_overall,
        "mean_cohen_d_sig": mean_d_overall,
        "n_sig_clusters": len(sig),
        "n_perm": n_perm,
        "alpha": alpha,
    }


# ─────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────

def run_assr_permutation_tests(groups=("TD", "ASD", "PMS"), n_perm=1000, alpha=0.05):
    """
    Extract group-averaged ITPC and ERSP maps, then run cluster-based
    permutation tests for all pairwise group comparisons.

    Comparisons: TD vs ASD, TD vs PMS, ASD vs PMS
    Features   : ITPC, ERSP

    Prints a structured results table to stdout and returns the full
    results dict for downstream use.
    """

    # ── 1. Collect features ───────────────────────────────────────────
    print("Collecting ASSR features …")
    data = {}
    for group in groups:
        ts_list, itpc_list, ersp_list = collate_ASSR_features(group)
        data[group] = {
            "ITPC": np.array(itpc_list),   # (n_subj, n_freqs, n_times)
            "ERSP": np.array(ersp_list),   # (n_subj, n_freqs, n_times)
        }
        print(f"  {group}: {data[group]['ITPC'].shape[0]} subjects, "
              f"ITPC shape {data[group]['ITPC'].shape[1:]}, "
              f"ERSP shape {data[group]['ERSP'].shape[1:]}")

    # ── 2. Define comparisons ─────────────────────────────────────────
    comparisons = list(combinations(groups, 2))   # (TD,ASD), (TD,PMS), (ASD,PMS)
    features    = ["ITPC", "ERSP"]

    all_results = {}

    # ── 3. Run tests ──────────────────────────────────────────────────
    sep = "=" * 72
    thin = "-" * 72

    print(f"\n{sep}")
    print(f"  CLUSTER-BASED PERMUTATION TESTS  |  n_perm={n_perm}  |  α={alpha}")
    print(sep)

    for feat in features:
        print(f"\n{'━'*72}")
        print(f"  Feature: {feat}")
        print(f"{'━'*72}")

        for g1, g2 in comparisons:
            key = f"{g1}_vs_{g2}_{feat}"
            print(f"\n  {g1}  vs  {g2}  [{feat}]")
            print(thin)

            A = data[g1][feat]
            B = data[g2][feat]

            res = cluster_permutation_test(A, B, n_perm=n_perm, alpha=alpha)
            all_results[key] = res

            clusters = res["observed_clusters"]
            if not clusters:
                print("  No supra-threshold clusters found.")
                continue

            # Header
            print(f"  {'Cluster':>7}  {'Sign':>4}  {'Size (px)':>9}  "
                  f"{'Mass (Σt)':>10}  {'p-value':>8}  "
                  f"{'Peak |d|':>9}  {'Mean |d|':>9}  {'Sig':>4}")
            print(f"  {'-'*7}  {'-'*4}  {'-'*9}  {'-'*10}  {'-'*8}  "
                  f"{'-'*9}  {'-'*9}  {'-'*4}")

            for r in sorted(clusters, key=lambda x: abs(x["mass"]), reverse=True):
                sig_flag = "***" if r["p_value"] < 0.001 else \
                           "**"  if r["p_value"] < 0.01  else \
                           "*"   if r["p_value"] < 0.05  else "n.s."
                print(f"  {r['cluster_id']:>7}  {r['sign']:>4}  "
                      f"{r['size']:>9}  {r['mass']:>10.2f}  "
                      f"{r['p_value']:>8.4f}  "
                      f"{r['peak_cohen_d']:>9.3f}  "
                      f"{r['mean_cohen_d']:>9.3f}  {sig_flag:>4}")

            n_sig = res["n_sig_clusters"]
            print(f"\n  Significant clusters (p < {alpha}): {n_sig}")
            if n_sig > 0:
                print(f"  Peak |Cohen's d| across sig. clusters : "
                      f"{res['max_cohen_d']:.3f}")
                print(f"  Mean |Cohen's d| across sig. pixels   : "
                      f"{res['mean_cohen_d_sig']:.3f}")

    print(f"\n{sep}\n")
    return all_results


# ─────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────
if __name__ == "__main__":
    results = run_assr_permutation_tests(
        groups=("TD", "ASD", "PMS"),
        n_perm=1000,
        alpha=0.05,
    )

In [ ]:
def plot_fooof_features(groups=["TD", "ASD", "PMS"]):
    """
    Updated FOOOF visualization:
      • r^2 and error -> boxplots
      • combined pairplot of aperiodic + periodic band features
      • FOOOF model fits on avg TS -> subplots
    """

    # ============================================================
    #   COLLECT FEATURES
    # ============================================================
    aperiodic_rows = []
    band_rows = []
    average_ts = {}
    peak_rows = []

    for group in groups:
        X, feat_names, avg_ts = collate_FOOOF_features(group)
        average_ts[group] = avg_ts

        # Feature structure:
        #   feat_names = ["r_squared", "error", "aperiodic_offset", "aperiodic_exponent", ...bands...]
        aper_names = feat_names[:4]
        band_names = feat_names[4:]

        aper_part = X[:, :4]
        band_part = X[:, 4:]

        # Store aperiodic
        for row in aper_part:
            aperiodic_rows.append(dict(zip(["Group"] + aper_names,
                                           [group] + list(row))))

        # Store band periodic
        for row in band_part:
            band_rows.append(dict(zip(["Group"] + band_names,
                                      [group] + list(row))))

        # Peak params for KDE or modeling
        sub_df = df[df["Group"] == group]
        for epoch_path in sub_df["Epoch Files"]:
            epoch = mne.read_epochs(epoch_path, preload=True)
            ts = get_ts(epoch)
            freqs, power = get_power_spectrum(ts)
            params = get_fooof_params(freqs, power)

            for c, p, bw in zip(params["peak_centers"],
                                params["peak_powers"],
                                params["peak_bandwidths"]):
                peak_rows.append({
                    "Group": group,
                    "Peak Center": c,
                    "Peak Power": p,
                    "Peak Bandwidth": bw
                })

    df_aper = pd.DataFrame(aperiodic_rows)
    df_band = pd.DataFrame(band_rows)
    df_peaks = pd.DataFrame(peak_rows)

    # ============================================================
    #   PANEL A: n_peaks, r_squared, error boxplots
    # ============================================================
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # n_peaks
    df_aper_with_n = df_aper.copy()
    df_aper_with_n["n_peaks"] = (
        df_band["theta_present"]
        + df_band["alpha_present"]
        + df_band["beta_present"]
        + df_band["gamma_present"]
    )

    sns.boxplot(data=df_aper_with_n, x="Group", y="n_peaks", ax=axes[0])
    axes[0].set_title("Number of Peaks")
    axes[0].set_ylabel("Number of Peaks")

    sns.boxplot(data=df_aper, x="Group", y="r_squared", ax=axes[1])
    axes[1].set_title("FOOOF r²")
    axes[1].set_ylabel("FOOOF r²")

    sns.boxplot(data=df_aper, x="Group", y="error", ax=axes[2])
    axes[2].set_title("FOOOF Error")
    axes[2].set_ylabel("FOOOF Error")

    fig.suptitle("FOOOF Fit Quality Metrics", fontsize=16)
    plt.tight_layout()
    plt.savefig("figures/1e.svg", format="svg", bbox_inches='tight')
    plt.show()

    # ============================================================
    #   PANEL B: Boxplots for Offset & Exponent (1 x 2) + significance
    # ============================================================

    import itertools
    from scipy.stats import mannwhitneyu, ttest_ind, shapiro

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    features = ["aperiodic_offset", "aperiodic_exponent"]
    titles = ["Aperiodic Offset", "Aperiodic Exponent"]

    def add_sig_bars(ax, data, feature):
        """Test normality → choose t-test or Mann–Whitney → annotate significance."""
        group_vals = {g: data[data["Group"] == g][feature].dropna() for g in groups}

        # Check normality for each group
        normal_flags = []
        for g in groups:
            vals = group_vals[g]
            if len(vals) < 3:
                normal_flags.append(False)
            else:
                _, p_norm = shapiro(vals)
                normal_flags.append(p_norm > 0.05)

        # True iff ALL groups are normally distributed
        use_ttest = all(normal_flags)

        pairs = list(itertools.combinations(groups, 2))

        y_max = data[feature].max()
        y_min = data[feature].min()
        h = (y_max - y_min) * 0.12
        start_y = y_max + 0.25 * h

        for (g1, g2) in pairs:
            x1 = groups.index(g1)
            x2 = groups.index(g2)

            xvals1 = group_vals[g1]
            xvals2 = group_vals[g2]

            if use_ttest:
                # Two-sample independent t-test
                _, p = ttest_ind(xvals1, xvals2, equal_var=False)
            else:
                # Mann–Whitney
                _, p = mannwhitneyu(xvals1, xvals2, alternative="two-sided")

            # Significance label
            if p < 0.001: sig = "***"
            elif p < 0.01: sig = "**"
            elif p < 0.05: sig = "*"
            else: sig = "n.s."

            ax.plot([x1, x1, x2, x2],
                    [start_y, start_y + h, start_y + h, start_y],
                    lw=1.5, color="k")

            ax.text((x1 + x2) / 2, start_y + h, sig,
                    ha="center", va="bottom", fontsize=14)

            start_y += h * 1.7

        ax.set_ylim(y_min, start_y + h)

    # --- Offset boxplot ---
    sns.boxplot(data=df_aper, x="Group", y=features[0], ax=axes[0])
    axes[0].set_title(titles[0], fontsize=14)
    axes[0].set_ylabel(titles[0])
    add_sig_bars(axes[0], df_aper, features[0])

    # --- Exponent boxplot ---
    sns.boxplot(data=df_aper, x="Group", y=features[1], ax=axes[1])
    axes[1].set_title(titles[1], fontsize=14)
    axes[1].set_ylabel(titles[1])
    add_sig_bars(axes[1], df_aper, features[1])

    fig.suptitle("Aperiodic Parameters Across Groups", fontsize=16)
    plt.tight_layout()
    plt.savefig("figures/1f.svg", format="svg", bbox_inches='tight')
    plt.show()

    # ============================================================
    #   PANEL C: FOOOF model fits on average TS
    # ============================================================
    fig, axes = plt.subplots(1, len(groups), figsize=(16, 4))

    for ax, group in zip(axes, groups):

        avg_ts = average_ts[group]
        freqs, power = compute_spectrum_welch(avg_ts, 250)

        fm = FOOOF(verbose=False)
        fm.fit(freqs, power, [2, 50])

        fm.plot(ax=ax, add_legend=False)
        ax.set_title(f"FOOOF Fit — {group}")

    fig.suptitle("FOOOF Model Fits on Mean Time Series", fontsize=16)
    plt.tight_layout()
    plt.savefig("figures/1d.svg", format="svg", bbox_inches='tight')
    plt.show()


In [ ]:
plot_fooof_features()

In [ ]:
df['Epochs'] = df['Epoch Files'].map(lambda x: mne.read_epochs(x,preload=True))
df['TS features'] = df['Epochs'].map(lambda x: get_ts(x))
df['ITPC features'] = df['Epochs'].map(lambda x: get_itpc_features(x).mean(axis=0))
df['ERSP features'] = df['Epochs'].map(lambda x: get_ersp_features(x).mean(axis=0))
df['FOOOF features'] = df['TS features'].map(lambda x: get_fooof_features(get_fooof_params(get_power_spectrum(x)[0],get_power_spectrum(x)[1]))[0])
df

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

def classifier_model(
        df,
        group1: str,
        group2: str,
        control_covariates: bool = True,
        random_state: int = 42
    ):
    """
    Train LOO-CV classifiers using any feature columns in df that end with 'features'.
    Optionally add age + sex as covariates to each submodel and to the combined model.
    If control_covariates=True, also fit a model that uses ONLY age+sex.
    """

    # ---------------------------------------------------------
    # Extract subjects
    # ---------------------------------------------------------
    sub_df1 = df[df["Group"] == group1].copy()
    sub_df2 = df[df["Group"] == group2].copy()
    sub_df = pd.concat([sub_df1, sub_df2], axis=0)

    labels = np.hstack([
        np.ones(len(sub_df1)),      
        np.zeros(len(sub_df2))      
    ])

    print(f"\nParticipants: {len(labels)} | {group1}={labels.sum()} | {group2}={(labels==0).sum()}")

    # ---------------------------------------------------------
    # Identify feature columns automatically
    # ---------------------------------------------------------
    feature_cols = [c for c in df.columns if c.endswith("features")]
    print("\nDetected feature sets:", feature_cols)

    # ---------------------------------------------------------
    # Convert each feature column into a (n_subjects, n_features) matrix
    # ---------------------------------------------------------
    feature_matrices = {}

    for col in feature_cols:
        feats = sub_df[col].to_numpy()
        try:
            X = np.vstack(feats)
        except Exception as e:
            print(f"Error stacking {col}: {e}")
            continue

        feature_matrices[col.replace(" features","")] = X

    # ---------------------------------------------------------
    # Add covariates (Age + Sex)
    # ---------------------------------------------------------
    if control_covariates:
        print("\nAdding covariates: Age + Sex")
        age = sub_df["Age_years"].to_numpy()[:, None]
        sex = sub_df["Sex"].map({"M":0,"F":1}).to_numpy()[:, None]
        
        covariates = np.hstack([age, sex])
        covariate_names = ["Age_years", "Sex"]
    else:
        covariates = None
        covariate_names = []

    # ---------------------------------------------------------
    # Prepare combined model
    # ---------------------------------------------------------
    combined_features = [X for X in feature_matrices.values()]
    if control_covariates:
        combined_features.append(covariates)
    combined_features = np.hstack(combined_features)

    # ---------------------------------------------------------
    # LOO setup
    # ---------------------------------------------------------
    def loo_predictions(X, y, model):
        if X.ndim != 2 or X.shape[1] == 0:
            return None, None, np.nan
        loo = LeaveOneOut()
        y_scores = cross_val_predict(
            model, X, y, cv=loo, method="predict_proba", n_jobs=1
        )[:, 1]
        auroc = roc_auc_score(y, y_scores)
        return y, y_scores, auroc

    # ---------------------------------------------------------
    # XGBoost model
    # ---------------------------------------------------------
    def make_clf():
        return XGBClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=random_state, n_jobs=4, eval_metric="logloss"
        )

    # ---------------------------------------------------------
    # Train individual models
    # ---------------------------------------------------------
    results = {}
    fitted_models = {}

    print("\n----- MODEL PERFORMANCE -----")

    for name, X in feature_matrices.items():

        if control_covariates:
            X = np.hstack([X, covariates])

        clf = make_clf()
        y_true, y_scores, aucv = loo_predictions(X, labels, clf)

        print(f"{name:>18s} | AUROC = {aucv:.3f}")

        results[name] = {"y_true": y_true, "y_scores": y_scores, "auroc": aucv}

        clf_full = make_clf()
        clf_full.fit(X, labels)
        fitted_models[name] = clf_full

    # ---------------------------------------------------------
    # NEW: Covariates-only model
    # ---------------------------------------------------------
    if control_covariates:
        print("\n--- Covariates-Only Model (Age + Sex) ---")

        X_cov = covariates  # only age+sex

        clf = make_clf()
        y_true, y_scores, aucv = loo_predictions(X_cov, labels, clf)

        print(f"CovariatesOnly | AUROC = {aucv:.3f}")

        results["CovariatesOnly"] = {
            "y_true": y_true,
            "y_scores": y_scores,
            "auroc": aucv,
        }

        clf_full = make_clf()
        clf_full.fit(X_cov, labels)
        fitted_models["CovariatesOnly"] = clf_full

    # ---------------------------------------------------------
    # Combined model
    # ---------------------------------------------------------
    print("\n--- Combined Feature Model ---")

    X_combined = combined_features
    clf = make_clf()

    y_true, y_scores, aucv = loo_predictions(X_combined, labels, clf)
    print(f"Combined Model | AUROC = {aucv:.3f}")

    results["Combined"] = {
        "y_true": y_true,
        "y_scores": y_scores,
        "auroc": aucv,
    }

    clf_full = make_clf()
    clf_full.fit(X_combined, labels)
    fitted_models["Combined"] = clf_full

    # ---------------------------------------------------------
    # Return everything
    # ---------------------------------------------------------
    return results, fitted_models


In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(results, title, filename):
    """
    Plot ROC curves for all models in the 'results' dictionary, with the highest
    AUROC highlighted. If a CovariatesOnly model exists, adjust legend labels so:
        • CovariatesOnly → "Age + Sex only"
        • All other models → "[model] + Age + Sex"
    """

    plt.figure(figsize=(7, 6))

    # Detect covariates-only model
    has_cov_only = "CovariatesOnly" in results

    # Colors
    colors = plt.cm.tab10.colors

    # --------------------------------------------------------
    # 1. Identify best-performing model
    # --------------------------------------------------------
    aucs = {
        name: data["auroc"]
        for name, data in results.items()
        if data.get("auroc") is not None
    }
    best_model = max(aucs, key=aucs.get)

    # --------------------------------------------------------
    # 2. Plot each ROC curve
    # --------------------------------------------------------
    for i, (model_name, data) in enumerate(results.items()):
        y_true = data.get("y_true")
        y_scores = data.get("y_scores")

        if y_true is None or y_scores is None:
            continue

        # Compute ROC curve
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        # Highlight best model visually
        alpha = 1.0 if model_name == best_model else 0.55
        linewidth = 2.5 if model_name == best_model else 1.2

        # --------------------------------------------------------
        # Legend formatting with special cases
        # --------------------------------------------------------
        if has_cov_only:
            # Covariates-only model
            if model_name == "CovariatesOnly":
                label = f"Age + Sex only (AUC = {roc_auc:.3f})"
            elif model_name == 'Combined':
                label = f"{model_name} (AUC = {roc_auc:.3f})"
            else:
                label = f"{model_name} + Age + Sex (AUC = {roc_auc:.3f})"
        else:
            # Normal behavior when covariates-only does not exist
            label = f"{model_name} (AUC = {roc_auc:.3f})"

        plt.plot(
            fpr,
            tpr,
            color=colors[i % len(colors)],
            linewidth=linewidth,
            alpha=alpha,
            label=label,
        )

    # --------------------------------------------------------
    # 3. Chance line
    # --------------------------------------------------------
    plt.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)

    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, format="svg", bbox_inches='tight')
    plt.show()


In [ ]:
results_TDPMS_WAS, fitted_models_TDPMS_WAS = classifier_model(df,'PMS','TD')

In [ ]:
plot_roc_curves(results_TDPMS_WAS, title='ROC Curves TD vs PMS with Age and Sex Covariates',filename='figures/2a.svg')

In [ ]:
results_TDPMS, fitted_models_TDPMS = classifier_model(df,'PMS','TD',control_covariates=False)

In [ ]:
plot_roc_curves(results_TDPMS, title = 'ROC Curves TD vs PMS without Covariates',filename='figures/S1a.svg')

In [ ]:
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score
)

def _compute_classification_metrics(y_true, y_scores, threshold=0.5): 
    """ Compute classification metrics with a fixed probability threshold. """ 
    y_pred = (y_scores >= threshold).astype(int) 
    sensitivity = recall_score(y_true, y_pred, pos_label=1) 
    specificity = recall_score(y_true, y_pred, pos_label=0) 
    precision = precision_score(y_true, y_pred, pos_label=1) 
    recall = recall_score(y_true, y_pred, pos_label=1) 
    f1 = f1_score(y_true, y_pred, pos_label=1) 
    return sensitivity, specificity, precision, recall, f1

def summarize_model_performance(
        results, 
        n_boot=1000, 
        ci=95, 
        threshold=0.5
    ):
    """
    Produce a formatted summary table (TS, ITPC, ERSP, FOOOF, Combined)
    with AUROC, Sensitivity, Specificity, F1, Precision including 95% CIs.
    Rounded to 2 decimals and copy/paste ready.
    """

    alpha = (100 - ci) / 2.0
    rows = []

    # ------------------------------------------------------------
    # Row order for final output
    # ------------------------------------------------------------
    desired_order = ["TS", "ITPC", "ERSP", "FOOOF", "Combined"]

    # Gather metrics
    for model_name, data in results.items():

        y_true = data["y_true"]
        y_scores = data["y_scores"]
        auc_val = data["auroc"]

        if y_true is None:
            continue

        n = len(y_true)

        # ============================================================
        # Point estimates
        # ============================================================
        sensitivity, specificity, precision, recall, f1 = \
            _compute_classification_metrics(y_true, y_scores, threshold)

        # ============================================================
        # Bootstrapping
        # ============================================================
        boot_aucs = []
        boot_sens = []
        boot_spec = []
        boot_prec = []
        boot_f1 = []

        for _ in range(n_boot):
            idx = np.random.randint(0, n, n)
            yt = y_true[idx]
            ys = y_scores[idx]

            boot_aucs.append(roc_auc_score(yt, ys))

            s, sp, pr, _, f = _compute_classification_metrics(yt, ys, threshold)
            boot_sens.append(s)
            boot_spec.append(sp)
            boot_prec.append(pr)
            boot_f1.append(f)

        boot_aucs = np.array(boot_aucs)
        boot_sens = np.array(boot_sens)
        boot_spec = np.array(boot_spec)
        boot_prec = np.array(boot_prec)
        boot_f1   = np.array(boot_f1)

        def CI(arr):
            return np.percentile(arr, alpha), np.percentile(arr, 100 - alpha)

        auc_low, auc_high = CI(boot_aucs)
        sens_low, sens_high = CI(boot_sens)
        spec_low, spec_high = CI(boot_spec)
        prec_low, prec_high = CI(boot_prec)
        f1_low,  f1_high  = CI(boot_f1)

        rows.append({
            "Model": model_name,
            "AUROC": (auc_val, auc_low, auc_high),
            "Sensitivity": (sensitivity, sens_low, sens_high),
            "Specificity": (specificity, spec_low, spec_high),
            "F1": (f1, f1_low, f1_high),
            "Precision": (precision, prec_low, prec_high)
        })

    df = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Enforce row order (only include rows that exist)
    # ------------------------------------------------------------
    df["order"] = df["Model"].apply(
        lambda x: desired_order.index(x) if x in desired_order else 999
    )
    df = df.sort_values("order").drop(columns=["order"])

    # ------------------------------------------------------------
    # Format each metric column as `mean (low–high)`
    # ------------------------------------------------------------
    def fmt(triple):
        mean, low, high = triple
        return f"{mean:.2f} ({low:.2f}–{high:.2f})"

    df["AUROC"] = df["AUROC"].apply(fmt)
    df["Sensitivity"] = df["Sensitivity"].apply(fmt)
    df["Specificity"] = df["Specificity"].apply(fmt)
    df["F1"] = df["F1"].apply(fmt)
    df["Precision"] = df["Precision"].apply(fmt)

    # ------------------------------------------------------------
    # Pretty print: this is the final table
    # ------------------------------------------------------------
    print("\n" + "="*95)
    print(f"{'MODEL PERFORMANCE SUMMARY':^95}")
    print("="*95)

    print(df.to_string(index=False))
    print("="*95 + "\n")

    return None


In [ ]:
summarize_model_performance(results_TDPMS_WAS)

In [ ]:
summarize_model_performance(results_TDPMS)

In [ ]:
def get_itpc_flattened(epoch: mne.Epochs, band_center=40, band_width=10):
    """Compute per-channel ITPC, average across channels manually."""
    fmin = band_center - band_width / 2.0
    fmax = band_center + band_width / 2.0
    freqs = np.linspace(fmin, fmax, 20)
    n_cycles = freqs / 2.0
    decim = 2

    itpc_list = []

    for ch_idx in range(len(epoch.ch_names)):
        _, itpc = mne.time_frequency.tfr_morlet(
            epoch, freqs=freqs, n_cycles=n_cycles, use_fft=True,
            return_itc=True, decim=decim, average=True, picks=[ch_idx]
        )
        itpc_list.append(itpc.data.squeeze())

    itpc_mean = np.mean(itpc_list, axis=0)
    
    return itpc_mean.flatten()

df['Flat ITPC'] = df['Epochs'].map(lambda x: get_itpc_flattened(x))
df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix

# ================================================================
# SETTINGS
# ================================================================
groups = ["TD", "ASD", "PMS"]
group_to_color = {"TD": "tab:blue", "ASD": "tab:green", "PMS": "tab:red"}
cluster_colors = {0: "grey", 1: "maroon"}

# ================================================================
# FEATURE MATRICES (FULL COHORT)
# ================================================================
X_ITPC = np.vstack(df["Flat ITPC"].to_numpy())

age = df["Age_years"].to_numpy()[:, None]
sex = df["Sex"].map({"M":0,"F":1}).to_numpy()[:, None]
cov = np.hstack([age, sex])

X_FULL = np.hstack([X_ITPC, cov])

true_labels = df["Group"].to_numpy()

# ================================================================
# FUNCTION: RUN UMAP + K-MEANS ON FULL COHORT
# ================================================================
def full_umap_and_kmeans(features):
    um = umap.UMAP(random_state=42)

    emb = um.fit_transform(features)

    km = KMeans(n_clusters=2, random_state=42, n_init="auto")
    clusters = km.fit_predict(emb)

    return emb, clusters

# Run models:
emb_ITPC, cl_ITPC   = full_umap_and_kmeans(X_ITPC)
emb_FULL, cl_FULL   = full_umap_and_kmeans(X_FULL)

# ================================================================
# PLOTTING HELPERS
# ================================================================
def plot_clusters(ax, emb, clusters, title):
    for c in [0, 1]:
        idx = np.where(clusters == c)[0]
        ax.scatter(
            emb[idx, 0], emb[idx, 1],
            c=cluster_colors[c], s=70,
            edgecolor="k", label=f"Cluster {c}"
        )
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend()

def plot_true_labels(ax, emb, labels, title):
    for g in groups:
        idx = np.where(labels == g)[0]
        ax.scatter(
            emb[idx, 0], emb[idx, 1],
            c=group_to_color[g], s=70,
            edgecolor="k", label=g
        )

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend()

# ================================================================
# BUILD 4 SEPARATE FIGURES
# ================================================================

# Figure 4a — ITPC + Age + Sex, K-means Clusters
fig, ax = plt.subplots(figsize=(9, 7))
plot_clusters(ax, emb_FULL, cl_FULL, "ITPC + Age + Sex — K-means Clusters")
plt.tight_layout()
plt.savefig("figures/5a.svg", format="svg", bbox_inches='tight')
plt.show()

# Figure 4b — ITPC + Age + Sex, True Group Labels
fig, ax = plt.subplots(figsize=(9, 7))
plot_true_labels(ax, emb_FULL, true_labels, "ITPC + Age + Sex — True Group Labels")
plt.tight_layout()
plt.savefig("figures/5b.svg", format="svg", bbox_inches='tight')
plt.show()

# Figure S3a — ITPC Only, K-means Clusters
fig, ax = plt.subplots(figsize=(9, 7))
plot_clusters(ax, emb_ITPC, cl_ITPC, "ITPC Only — K-means Clusters")
plt.tight_layout()
plt.savefig("figures/S4a.svg", format="svg", bbox_inches='tight')
plt.show()

# Figure S3b — ITPC Only, True Group Labels
fig, ax = plt.subplots(figsize=(9, 7))
plot_true_labels(ax, emb_ITPC, true_labels, "ITPC Only — True Group Labels")
plt.tight_layout()
plt.savefig("figures/S4b.svg", format="svg", bbox_inches='tight')
plt.show()

# ================================================================
# PMS CLASSIFICATION PERFORMANCE (FULL COHORT)
# ================================================================
def compute_pms_performance(cluster_labels, true_labels):
    mask = np.isin(true_labels, ["TD", "PMS"])
    binary_true = (true_labels[mask] == "PMS").astype(int)
    cl = cluster_labels[mask]

    results = {}
    for cluster_id in [0, 1]:
        pred = (cl == cluster_id).astype(int)

        tn, fp, fn, tp = confusion_matrix(binary_true, pred).ravel()

        sens = tp/(tp+fn) if (tp+fn)>0 else np.nan
        spec = tn/(tn+fp) if (tn+fp)>0 else np.nan

        results[f"Cluster {cluster_id}"] = (sens, spec)

    return results

# ITPC only
perf_itpc = compute_pms_performance(cl_ITPC, true_labels)

# ITPC + Age + Sex
perf_full = compute_pms_performance(cl_FULL, true_labels)

print("\n=======================================================")
print(" PMS Classification Based on Clusters")
print("=======================================================\n")

print("ITPC ONLY model:")
for cl, (sens, spec) in perf_itpc.items():
    print(f"  {cl}: Sensitivity={sens:.3f}, Specificity={spec:.3f}")
print()

print("ITPC + Age + Sex model:")
for cl, (sens, spec) in perf_full.items():
    print(f"  {cl}: Sensitivity={sens:.3f}, Specificity={spec:.3f}")
print()


In [ ]:
df["UMAP1_ITPC"] = emb_ITPC[:, 0]
df["UMAP2_ITPC"] = emb_ITPC[:, 1]
df["UMAP1_FULL"] = emb_FULL[:, 0]
df["UMAP2_FULL"] = emb_FULL[:, 1]

df["Cluster_ITPC"] = cl_ITPC
df["Cluster_FULL"] = cl_FULL

df['ITPC mean'] = df['ITPC features'].map(lambda x: x[63:126].mean())

# ================================================================
# ADD DISTANCE TO CLUSTER-1 CENTROID (ITPC + FULL)
# ================================================================

# Compute centroids of cluster 1
centroid_ITPC_1 = emb_ITPC[cl_ITPC == 1].mean(axis=0)
centroid_FULL_1 = emb_FULL[cl_FULL == 1].mean(axis=0)

# Compute Euclidean distances for all subjects
df["Dist_to_Cluster1_ITPC"] = np.linalg.norm(
    emb_ITPC - centroid_ITPC_1, axis=1
)

df["Dist_to_Cluster1_FULL"] = np.linalg.norm(
    emb_FULL - centroid_FULL_1, axis=1
)

# ================================================================
# Subset ASD (safe)
# ================================================================
ASD_df = df[df['Group']=='ASD'].copy()

# Extract features + covariates
ASD_features = np.vstack(ASD_df["ITPC features"].to_numpy())
ASD_age  = ASD_df["Age_years"].to_numpy()[:, None]
ASD_sex  = ASD_df["Sex"].map({"M":0, "F":1}).to_numpy()[:, None]
ASD_cov  = np.hstack([ASD_age, ASD_sex])

# Predict PMS-likeness
pms_like_ASD_ITPC = fitted_models_TDPMS["ITPC"].predict_proba(ASD_features)[:, 1]
ASD_full = np.hstack([ASD_features, ASD_cov])
pms_like_ASD_FULL = fitted_models_TDPMS_WAS["ITPC"].predict_proba(ASD_full)[:, 1]

# Safe assignment
ASD_df['PMS_Likeness_ITPC'] = pms_like_ASD_ITPC
ASD_df['PMS_Likeness_FULL'] = pms_like_ASD_FULL


In [ ]:
from scipy.stats import gaussian_kde
# ─────────────────────────────────────────────
# Data
# ─────────────────────────────────────────────
labels = results_TDPMS_WAS['ITPC']['y_true'].astype(int)
scores = results_TDPMS_WAS['ITPC']['y_scores']

PMS_scores = scores[labels == 1]
TD_scores  = scores[labels == 0]
ASD_scores = ASD_df['PMS_Likeness_ITPC'].dropna().values

group_data   = [TD_scores, ASD_scores, PMS_scores]
group_labels = ['TD', 'ASD', 'PMS']
group_colors = ['#2196F3', '#4CAF50', '#F44336']  # blue, green, red

# ─────────────────────────────────────────────
# Helper
# ─────────────────────────────────────────────
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

# ─────────────────────────────────────────────
# Plot 1 — Boxplot (no color, significance stars)
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4.5))

bp = ax.boxplot(
    group_data,
    labels=group_labels,
    patch_artist=True,
    widths=0.40,
    medianprops=dict(linewidth=2),
    whiskerprops=dict(linewidth=1.3),
    capprops=dict(linewidth=1.3),
    flierprops=dict(marker='o', markersize=3.5, alpha=0.6),
    boxprops=dict(linewidth=1.3),
)

ax.set_ylabel('PMS-Likeness Score', fontsize=11)
ax.set_title('PMS-Likeness in Model w/ Covariates by Group', fontsize=12, pad=10)

# --- compute all three p-values first so we can set ylim precisely ---
pairs = [(0, 1), (0, 2), (1, 2)]   # TD-ASD, TD-PMS, ASD-PMS
x_pos = [1, 2, 3]

pvals = {}
for i, j in pairs:
    _, p = stats.ttest_ind(group_data[i], group_data[j], equal_var=False)
    pvals[(i, j)] = p

# bracket geometry anchored to data ceiling, not axes ylim
data_max = max(np.max(g) for g in group_data)

gap  = 0.06   # space between data and first bracket
step = 0.07   # space between bracket levels

# levels: 0 = lowest, 1 = middle, 2 = highest
bracket_levels = {(0, 1): 0, (1, 2): 1, (0, 2): 2}
y_brackets     = {k: data_max + gap + v * step for k, v in bracket_levels.items()}
y_top          = data_max + gap + 2 * step + step * 0.6

ax.set_ylim(ax.get_ylim()[0], y_top)

bar_tip = step * 0.18

for (i, j), y in y_brackets.items():
    label = sig_label(pvals[(i, j)])
    x1, x2 = x_pos[i], x_pos[j]
    ax.plot([x1, x1, x2, x2],
            [y - bar_tip, y, y, y - bar_tip],
            color='black', linewidth=1.1)
    ax.text((x1 + x2) / 2, y + step * 0.04,
            label,
            ha='center', va='bottom',
            fontsize=11 if label != 'n.s.' else 9,
            fontstyle='italic' if label == 'n.s.' else 'normal')

sns.despine(ax=ax, left=False)
plt.tight_layout()
plt.savefig("figures/3a.svg", format="svg", bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────
# Plot 2 — KDE (colored per group)
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

x_grid = np.linspace(0, 1, 500)

for scores_grp, label, color in zip(group_data, group_labels, group_colors):
    n = len(scores_grp)
    kde = gaussian_kde(scores_grp, bw_method='scott')
    density = kde(x_grid)
    ax.plot(x_grid, density, color=color, linewidth=2.2, label=f'{label} (n={n})')
    ax.fill_between(x_grid, density, alpha=0.15, color=color)

ax.axvline(0.5, color='dimgrey', linestyle='--', linewidth=1.3, label='Threshold (0.5)')

ax.set_xlabel('PMS Probability Score', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of PMS Scores (w/ Covariates) Across Groups', fontsize=12, pad=10)
ax.set_xlim(0, 1)
ax.legend(framealpha=0.6, fontsize=10)

sns.despine(ax=ax)
plt.tight_layout()
plt.savefig("figures/3b.svg", format="svg", bbox_inches='tight')
plt.show()

# Get ASD scores and drop NaNs if any
ASD_scores = ASD_df['PMS_Likeness_FULL'].dropna()

# Sort from highest to lowest
ASD_sorted = np.sort(ASD_scores)[::-1]

# Create subject numbers 1 → N (should be 56)
subjects = np.arange(1, len(ASD_sorted) + 1)

plt.figure(figsize=(10, 6))
plt.bar(subjects, ASD_sorted)
# Add dashed threshold line at 0.5
plt.axhline(y=0.5, linestyle='--')

plt.xlabel('Subject Number (Ranked)')
plt.ylabel('PMS-Likeness')
plt.title('ASD Participants Ranked by PMS-Likeness w/ Covariates')
plt.xticks(subjects, rotation=90)


plt.xticks(subjects)  # will show 1–56 if length is 56
plt.tight_layout()
plt.savefig("figures/3c.svg", format="svg", bbox_inches='tight')
plt.show()

In [ ]:


sns.set_theme(style="white", font_scale=1.05)

# ─────────────────────────────────────────────
# Data
# ─────────────────────────────────────────────
labels = results_TDPMS['ITPC']['y_true'].astype(int)
scores = results_TDPMS['ITPC']['y_scores']

PMS_scores = scores[labels == 1]
TD_scores  = scores[labels == 0]
ASD_scores = ASD_df['PMS_Likeness_ITPC'].dropna().values

group_data   = [TD_scores, ASD_scores, PMS_scores]
group_labels = ['TD', 'ASD', 'PMS']
group_colors = ['#2196F3', '#4CAF50', '#F44336']  # blue, green, red

# ─────────────────────────────────────────────
# Helper
# ─────────────────────────────────────────────
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

# ─────────────────────────────────────────────
# Plot 1 — Boxplot (no color, significance stars)
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4.5))

bp = ax.boxplot(
    group_data,
    labels=group_labels,
    patch_artist=True,
    widths=0.40,
    medianprops=dict(linewidth=2),
    whiskerprops=dict(linewidth=1.3),
    capprops=dict(linewidth=1.3),
    flierprops=dict(marker='o', markersize=3.5, alpha=0.6),
    boxprops=dict(linewidth=1.3),
)

ax.set_ylabel('PMS-Likeness Score', fontsize=11)
ax.set_title('PMS-Likeness in Model w/o Covariates by Group', fontsize=12, pad=10)

# --- compute all three p-values first so we can set ylim precisely ---
pairs = [(0, 1), (0, 2), (1, 2)]   # TD-ASD, TD-PMS, ASD-PMS
x_pos = [1, 2, 3]

pvals = {}
for i, j in pairs:
    _, p = stats.ttest_ind(group_data[i], group_data[j], equal_var=False)
    pvals[(i, j)] = p

# bracket geometry anchored to data ceiling, not axes ylim
data_max = max(np.max(g) for g in group_data)

gap  = 0.06   # space between data and first bracket
step = 0.07   # space between bracket levels

# levels: 0 = lowest, 1 = middle, 2 = highest
bracket_levels = {(0, 1): 0, (1, 2): 1, (0, 2): 2}
y_brackets     = {k: data_max + gap + v * step for k, v in bracket_levels.items()}
y_top          = data_max + gap + 2 * step + step * 0.6

ax.set_ylim(ax.get_ylim()[0], y_top)

bar_tip = step * 0.18

for (i, j), y in y_brackets.items():
    label = sig_label(pvals[(i, j)])
    x1, x2 = x_pos[i], x_pos[j]
    ax.plot([x1, x1, x2, x2],
            [y - bar_tip, y, y, y - bar_tip],
            color='black', linewidth=1.1)
    ax.text((x1 + x2) / 2, y + step * 0.04,
            label,
            ha='center', va='bottom',
            fontsize=11 if label != 'n.s.' else 9,
            fontstyle='italic' if label == 'n.s.' else 'normal')

sns.despine(ax=ax, left=False)
plt.tight_layout()
plt.savefig("figures/S2a.svg", format="svg", bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────
# Plot 2 — KDE (colored per group)
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

x_grid = np.linspace(0, 1, 500)

for scores_grp, label, color in zip(group_data, group_labels, group_colors):
    n = len(scores_grp)
    kde = gaussian_kde(scores_grp, bw_method='scott')
    density = kde(x_grid)
    ax.plot(x_grid, density, color=color, linewidth=2.2, label=f'{label} (n={n})')
    ax.fill_between(x_grid, density, alpha=0.15, color=color)

ax.axvline(0.5, color='dimgrey', linestyle='--', linewidth=1.3, label='Threshold (0.5)')

ax.set_xlabel('PMS Probability Score', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of PMS Scores (w/o Covariates) Across Groups', fontsize=12, pad=10)
ax.set_xlim(0, 1)
ax.legend(framealpha=0.6, fontsize=10)

sns.despine(ax=ax)
plt.tight_layout()
plt.savefig("figures/S2b.svg", format="svg", bbox_inches='tight')
plt.show()


# Get ASD scores and drop NaNs if any
ASD_scores = ASD_df['PMS_Likeness_ITPC'].dropna()

# Sort from highest to lowest
ASD_sorted = np.sort(ASD_scores)[::-1]

# Create subject numbers 1 → N (should be 56)
subjects = np.arange(1, len(ASD_sorted) + 1)

plt.figure(figsize=(10, 6))
plt.bar(subjects, ASD_sorted)
# Add dashed threshold line at 0.5
plt.axhline(y=0.5, linestyle='--')

plt.xlabel('Subject Number (Ranked)')
plt.ylabel('PMS-Likeness')
plt.title('ASD Participants Ranked by PMS-Likeness w/o Covariates')
plt.xticks(subjects, rotation=90)


plt.xticks(subjects)  # will show 1–56 if length is 56
plt.tight_layout()
plt.savefig("figures/S2c.svg", format="svg", bbox_inches='tight')
plt.show()



In [ ]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# HELPERS
# ============================================================

def r_confidence_interval(r, n, alpha=0.05):
    if n < 4:
        return np.nan, np.nan
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    zcrit = stats.norm.ppf(1 - alpha/2)
    return np.tanh(z - zcrit*se), np.tanh(z + zcrit*se)

CLUSTER_COLORS = {0: "grey", 1: "maroon"}

def make_legend(r2, p):
    return [
        Line2D([0], [0], marker='o', color=CLUSTER_COLORS[0],
               markersize=8, linestyle="", label='Cluster 0'),
        Line2D([0], [0], marker='o', color=CLUSTER_COLORS[1],
               markersize=8, linestyle="", label='Cluster 1'),
        Line2D([0], [0], color="black", lw=2,
               label=f"Regression\nR²={r2:.2f}, p={p:.3g}")
    ]

def scatter_regression_ax(ax, X, Y, C):
    """Plot scatter + regression line + CI on a given axis. Returns r, p."""
    mask = (~X.isna()) & (~Y.isna())
    X, Y, C = X[mask], Y[mask], C[mask]

    if len(X) < 3:
        return np.nan, np.nan

    r, p   = stats.pearsonr(X, Y)
    slope, intercept, *_ = stats.linregress(X, Y)

    ax.scatter(X, Y, alpha=0.75, color=C, edgecolor="k")

    xvals  = np.linspace(X.min(), X.max(), 200)
    y_pred = slope * xvals + intercept
    y_fit  = slope * X + intercept
    resid  = Y - y_fit
    dof    = len(X) - 2
    tcrit  = stats.t.ppf(0.975, dof)
    mean_x = X.mean()
    s_xx   = np.sum((X - mean_x)**2)
    se_pred = np.sqrt(
        (np.sum(resid**2) / dof) *
        (1/len(X) + (xvals - mean_x)**2 / s_xx)
    )

    ax.fill_between(xvals, y_pred - tcrit*se_pred,
                           y_pred + tcrit*se_pred,
                    color="darkgrey", alpha=0.2)
    ax.plot(xvals, y_pred, color="black", linewidth=2)
    ax.grid(False)
    ax.legend(handles=make_legend(r**2, p), fontsize=9)

    return r, p


def save_scatter(xcol, ycol, cluster_col, xlab, ylab, filepath, ASD):
    """Create and save a single scatter panel."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ASD = ASD.copy()
    ASD["plot_color"] = ASD[cluster_col].map(CLUSTER_COLORS)
    scatter_regression_ax(ax, ASD[xcol], ASD[ycol], ASD["plot_color"])
    ax.set_xlabel(xlab, fontsize=12)
    ax.set_ylabel(ylab, fontsize=12)
    plt.tight_layout()
    plt.savefig(filepath, format="svg", bbox_inches='tight')
    plt.show()
    plt.close()


def save_distance_panel(xcol, ycol, cluster_col, xlab, ylab, filepath, ASD):
    """Create and save the wider cluster-distance panel."""
    fig, ax = plt.subplots(figsize=(12, 5))
    ASD = ASD.copy()
    ASD["plot_color"] = ASD[cluster_col].map(CLUSTER_COLORS)
    scatter_regression_ax(ax, ASD[xcol], ASD[ycol], ASD["plot_color"])
    ax.set_xlabel(xlab, fontsize=12)
    ax.set_ylabel(ylab, fontsize=12)
    plt.tight_layout()
    plt.savefig(filepath, format="svg", bbox_inches='tight')
    plt.show()
    plt.close()


ASD = ASD_df.copy()

# ============================================================
# FIGURE 6 — ITPC + Age/Sex model (main figures)
# ============================================================

# 6a — Age
save_scatter(
    "Age_years", "PMS_Likeness_FULL", "Cluster_FULL",
    "Age (years)", "SAI (ITPC + Age/Sex)",
    "figures/6a.svg", ASD
)

# 6b — IQ
save_scatter(
    "IQ_score", "PMS_Likeness_FULL", "Cluster_FULL",
    "IQ Score", "SAI (ITPC + Age/Sex)",
    "figures/6b.svg", ASD
)

# 6c — ADOS Total
save_scatter(
    "ADOS_comp_score", "PMS_Likeness_FULL", "Cluster_FULL",
    "ADOS Comparison Score", "SAI (ITPC + Age/Sex)",
    "figures/6c.svg", ASD
)

# 6d — ITPC Mean
save_scatter(
    "ITPC mean", "PMS_Likeness_FULL", "Cluster_FULL",
    "ITPC Mean (.5 s post onset)", "SAI (ITPC + Age/Sex)",
    "figures/6d.svg", ASD
)

# 6e — Cluster distance (wide panel)
save_distance_panel(
    "Dist_to_Cluster1_FULL", "PMS_Likeness_FULL", "Cluster_FULL",
    "Distance to Cluster 1 (Full UMAP space)", "SAI (ITPC + Age/Sex)",
    "figures/6e.svg", ASD
)

# ============================================================
# FIGURES S5a-S5e — ITPC only model (supplemental)
# ============================================================

# S5a — Age
save_scatter(
    "Age_years", "PMS_Likeness_ITPC", "Cluster_ITPC",
    "Age (years)", "SAI (ITPC only)",
    "figures/S5a.svg", ASD
)

# S5b — IQ
save_scatter(
    "IQ_score", "PMS_Likeness_ITPC", "Cluster_ITPC",
    "IQ Score", "SAI (ITPC only)",
    "figures/S5b.svg", ASD
)

# S5c — ADOS Total
save_scatter(
    "ADOS_comp_score", "PMS_Likeness_ITPC", "Cluster_ITPC",
    "ADOS Comparison Score", "SAI (ITPC only)",
    "figures/S5c.svg", ASD
)

# S5d — ITPC Mean
save_scatter(
    "ITPC mean", "PMS_Likeness_ITPC", "Cluster_ITPC",
    "ITPC Mean (.5 s post onset)", "SAI (ITPC only)",
    "figures/S5d.svg", ASD
)

# S5e — Cluster distance (wide panel)
save_distance_panel(
    "Dist_to_Cluster1_ITPC", "PMS_Likeness_ITPC", "Cluster_ITPC",
    "Distance to Cluster 1 (ITPC UMAP space)", "SAI (ITPC only)",
    "figures/S5e.svg", ASD
)

# ============================================================
# PRINT RESULTS
# ============================================================

x_vars = [
    ("Age_years",        "Age (years)"),
    ("IQ_score",         "IQ Score"),
    ("ADOS_comp_score", "ADOS Comparison Score"),
    ("ITPC mean",        "ITPC Mean (.5 s post onset)"),
]

y_vars = [
    ("PMS_Likeness_ITPC", "SAI (ITPC only)",       "Cluster_ITPC"),
    ("PMS_Likeness_FULL", "SAI (ITPC + Age/Sex)",  "Cluster_FULL"),
]

print("\n==============================================================")
print(" ASD-Only Correlation Results")
print("==============================================================\n")

for ycol, ylab, cluster_col in y_vars:
    print(f"\n====================== {ylab} ======================")
    ASD["plot_color"] = ASD[cluster_col].map(CLUSTER_COLORS)

    for xcol, xlab in x_vars:
        X = ASD[xcol]
        Y = ASD[ycol]
        mask = (~X.isna()) & (~Y.isna())
        X, Y = X[mask], Y[mask]

        if len(X) < 3:
            continue

        r, p   = stats.pearsonr(X, Y)
        r2     = r**2
        r_low, r_high = r_confidence_interval(r, len(X))
        slope, intercept, *_ = stats.linregress(X, Y)

        print(f"\n--- {xlab} ---")
        print(f"  r  = {r:.3f}")
        print(f"  p  = {p:.3g}")
        print(f"  r² = {r2:.3f}")
        print(f"  95% CI for r = [{r_low:.3f}, {r_high:.3f}]")
        print(f"  Line: y = {slope:.4f} * x + {intercept:.4f}")
        print(f"  n  = {len(X)}")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Start from your existing ASD_df
ASD = ASD_df.copy()

# Rename the column with a space so we can use it easily in formulas
ASD = ASD.rename(columns={"ITPC mean": "ITPC_mean"})

# List of predictors (matching your x_vars)
predictors = ["Age_years", "IQ_score", "ITPC_mean"]

# Make sure we only use rows with complete data for all relevant variables
def clean_data(df, outcome, predictors):
    cols = [outcome] + predictors
    return df[cols].dropna()

# Helper to fit model, print ANOVA + coefficients
def fit_and_report(outcome, predictors, data, model_label=""):
    formula = outcome + " ~ " + " + ".join(predictors)
    print("\n" + "="*70)
    print(f"Multiple regression: {outcome} ~ " + " + ".join(predictors))
    if model_label:
        print(f"Model label: {model_label}")
    print("="*70 + "\n")

    # Drop rows with any missing values in the model variables
    df_clean = clean_data(data, outcome, predictors)

    # Fit OLS model
    model = smf.ols(formula, data=df_clean).fit()

    # --- ANOVA table (Type II or III; II is common if you don't have interactions) ---
    anova_table = sm.stats.anova_lm(model, typ=2)

    print("ANOVA table (Type II):")
    print(anova_table)
    print("\n")

    # --- Coefficients with t-tests and p-values ---
    print("Coefficient table:")
    print(model.summary().tables[1])  # This is the table of coefficients
    print("\n")

    # Optional: print n used
    print(f"Number of observations used: {int(model.nobs)}")
    print("\n")

    return model, anova_table

# ---------------------------------------------------
# 1) PMS_Likeness_ITPC as outcome
# ---------------------------------------------------
model_itpc, anova_itpc = fit_and_report(
    outcome="PMS_Likeness_ITPC",
    predictors=predictors,
    data=ASD,
    model_label="PMS-Likeness (ITPC only)"
)

# ---------------------------------------------------
# 2) PMS_Likeness_FULL as outcome
# ---------------------------------------------------
model_full, anova_full = fit_and_report(
    outcome="PMS_Likeness_FULL",
    predictors=predictors,
    data=ASD,
    model_label="PMS-Likeness (ITPC + Age/Sex)"
)


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Copy data and rename ITPC column (if not already done)
ASD = ASD_df.copy()
ASD = ASD.rename(columns={"ITPC mean": "ITPC_mean"})

predictors = ["Age_years", "IQ_score", "ADOS_comp_score", "ITPC_mean"]

# Drop rows with missing data
X = ASD[predictors].dropna()

# Add intercept (required for VIF calculation in statsmodels)
X_const = sm.add_constant(X)

# Compute VIF
vif_table = pd.DataFrame({
    "Variable": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i)
            for i in range(X_const.shape[1])]
})

print("\n================ VIF TABLE ================\n")
print(vif_table)


In [ ]:
from sklearn.preprocessing import StandardScaler

X_z = pd.DataFrame(
    StandardScaler().fit_transform(X),
    columns=X.columns,
    index=X.index
)

Xz_const = sm.add_constant(X_z)
_, svals_z, _ = np.linalg.svd(Xz_const.values)
condition_number_z = svals_z[0] / svals_z[-1]

print("\n=========== CONDITION NUMBER (STANDARDIZED) ===========\n")
print(f"Standardized condition number: {condition_number_z:.2f}")

In [ ]:
results_ASDPMS_WAS, fitted_models_ASDPMS_WAS = classifier_model(df,'PMS','ASD')

In [ ]:
plot_roc_curves(results_ASDPMS_WAS, title='ROC Curves ASD [Full Cohort] vs PMS \n with Age and Sex Covariates', filename='figures/4a.svg')

In [ ]:
summarize_model_performance(results_ASDPMS_WAS)

In [ ]:
results_ASDTD_WAS, fitted_models_ASDTD_WAS = classifier_model(df,'TD','ASD')

In [ ]:
plot_roc_curves(results_ASDTD_WAS, title='ROC Curves ASD [Full Cohort] vs TD \n with Age and Sex Covariates', filename='figures/4c.svg')

In [ ]:
summarize_model_performance(results_ASDTD_WAS)

In [ ]:
results_ASDPMS, fitted_models_ASDPMS = classifier_model(df,'PMS','ASD',control_covariates=False)

In [ ]:
plot_roc_curves(results_ASDPMS, title='ROC Curves ASD [Full Cohort] vs PMS \n without Covariates',filename='figures/S3a.svg')

In [ ]:
summarize_model_performance(results_ASDPMS)

In [ ]:
results_ASDTD, fitted_models_ASDTD = classifier_model(df,'TD','ASD',control_covariates=False)

In [ ]:
plot_roc_curves(results_ASDTD, title='ROC Curves ASD [Full Cohort] vs TD \n without Covariates',filename='figures/S3c.svg')

In [ ]:
summarize_model_performance(results_ASDTD)

In [ ]:
merged_df = df.merge(right=ASD_df[['Participant ID','PMS_Likeness_ITPC','PMS_Likeness_FULL']],
              left_on='Participant ID',right_on='Participant ID',
              how='left')
merged_df

In [ ]:
labels = results_TDPMS['ITPC']['y_true'].astype(int)
scores = results_TDPMS['ITPC']['y_scores']
PMS_scores = scores[labels ==1]
TD_scores = scores[labels == 0]
ASD_scores = merged_df

In [ ]:
low_SAI_df = merged_df[~((merged_df['Group'] == 'ASD') & (merged_df['PMS_Likeness_FULL'] >= .5))]
high_SAI_df = merged_df[~((merged_df['Group'] == 'ASD') & (merged_df['PMS_Likeness_FULL'] < .5))]

In [ ]:
#we want to do the following comparisons: 
#fig 4b low_SAI ASD vs PMS w/ covariates
#fig 4d high_SAI ASD vs TD w/ covariates
#fig S3b low_SAI ASD vs PMS w/ covariates
#fig S3d high_SAI ASD vs TD w/ covariates

In [ ]:
results_LSAI_ASDPMS_WAS, fitted_models_LSAI_ASDPMS_WAS = classifier_model(low_SAI_df,'PMS','ASD')

In [ ]:
plot_roc_curves(results_LSAI_ASDPMS_WAS, title='ROC Curves ASD [Low SAI Cohort] vs PMS \n with Covariates',filename='figures/4b.svg')

In [ ]:
summarize_model_performance(results_LSAI_ASDPMS_WAS)

In [ ]:
results_HSAI_ASDTD_WAS, fitted_models_HSAI_ASDTD_WAS = classifier_model(high_SAI_df,'TD','ASD')

In [ ]:
plot_roc_curves(results_HSAI_ASDTD_WAS, title='ROC Curves ASD [High SAI Cohort] vs TD \n with Covariates',filename='figures/4d.svg')

In [ ]:
summarize_model_performance(results_HSAI_ASDTD_WAS)

In [ ]:
results_LSAI_ASDPMS, fitted_models_LSAI_ASDPMS = classifier_model(low_SAI_df,'PMS','ASD',control_covariates=False)

In [ ]:
plot_roc_curves(results_LSAI_ASDPMS, title='ROC Curves ASD [Low SAI Cohort] vs PMS \n without Covariates',filename='figures/S3b.svg')

In [ ]:
summarize_model_performance(results_LSAI_ASDPMS)

In [ ]:
results_HSAI_ASDTD, fitted_models_HSAI_ASDTD = classifier_model(high_SAI_df,'TD','ASD',control_covariates=False)

In [ ]:
plot_roc_curves(results_HSAI_ASDTD, title='ROC Curves ASD [High SAI Cohort] vs TD \n without Covariates',filename='figures/S3d.svg')

In [ ]:
summarize_model_performance(results_HSAI_ASDTD)

In [ ]:
import numpy as np
import pandas as pd

# --- ITPC MODEL ---
# Compute terciles
itpc_bins = pd.qcut(
    merged_df['PMS_Likeness_ITPC'],
    q=3,
    labels=[0, 1, 2]   # bottom=0, middle=NA, top=1
)

merged_df['Group2 ITPC'] = itpc_bins

# --- FULL MODEL ---
full_bins = pd.qcut(
    merged_df['PMS_Likeness_FULL'],
    q=3,
    labels=[0, 1, 2]
)

merged_df['Group2 FULL'] = full_bins


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


def classifier_group2_from_asd(
        df,
        group_asd: str = "ASD",
        group_pms: str = "PMS",
        group_td: str = "TD",
        group2_col: str = "Group2 FULL",
        control_covariates: bool = True,
        random_state: int = 42,
        n_label_shuffles: int = 100,
    ):
    """
    Train LOO-CV classifiers on ASD-only participants to distinguish Group2 0 vs 2
    (middle tertile = 1 is excluded), then test the trained models on unseen PMS vs TD.

    Includes a label-shuffled null distribution analysis on the PMS vs TD test set:
    the trained model is held fixed and PMS/TD labels are randomly permuted
    `n_label_shuffles` times to estimate the expected AUROC under the null hypothesis
    that group membership is uninformative. A permutation p-value is reported as the
    proportion of shuffled AUROCs >= the observed AUROC.

    Parameters
    ----------
    df                : merged dataframe with all groups
    group_asd         : label for ASD group in 'Group' column
    group_pms         : label for PMS group in 'Group' column
    group_td          : label for TD group in 'Group' column
    group2_col        : column name for ASD tertile labels (0, 1, 2)
    control_covariates: whether to include Age + Sex as covariates
    random_state      : random seed for XGBoost and shuffle RNG
    n_label_shuffles  : number of label-shuffle iterations for null distribution

    Returns
    -------
    results       : dict[model_name] -> dict with ASD and PMSvsTD performance + shuffle null
    fitted_models : dict[model_name] -> trained XGBClassifier
    """

    rng = np.random.default_rng(random_state)

    # ─────────────────────────────────────────────
    # Helpers
    # ─────────────────────────────────────────────
    def make_clf():
        return XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=4,
            eval_metric="logloss"
        )

    def loo_predictions(X, y, model):
        if X.ndim != 2 or X.shape[1] == 0:
            return None, None, np.nan
        loo = LeaveOneOut()
        y_scores = cross_val_predict(
            model, X, y, cv=loo, method="predict_proba", n_jobs=1
        )[:, 1]
        auroc = roc_auc_score(y, y_scores)
        return y, y_scores, auroc

    def shuffle_null(clf_full, X_test, y_test, n_shuffles, rng):
        """
        Hold the trained model fixed. Shuffle PMS/TD labels n_shuffles times
        and record AUROC each time to build a null distribution.

        Returns
        -------
        null_aurocs : np.ndarray, shape (n_shuffles,)
        null_mean   : float
        null_std    : float
        p_value     : proportion of null AUROCs >= observed AUROC
        """
        observed_auroc = roc_auc_score(y_test, clf_full.predict_proba(X_test)[:, 1])
        null_aurocs = np.zeros(n_shuffles)

        for i in range(n_shuffles):
            y_shuffled = rng.permutation(y_test)
            null_aurocs[i] = roc_auc_score(y_shuffled,
                                           clf_full.predict_proba(X_test)[:, 1])

        p_value = float(np.mean(null_aurocs >= observed_auroc))
        return null_aurocs, float(null_aurocs.mean()), float(null_aurocs.std()), p_value

    # ─────────────────────────────────────────────
    # Feature columns
    # ─────────────────────────────────────────────
    feature_cols = [c for c in df.columns if c.endswith("features")]
    print("\nDetected feature sets:", feature_cols)

    # ─────────────────────────────────────────────
    # ASD training set — Group2 extremes only
    # ─────────────────────────────────────────────
    asd_df = df[df["Group"] == group_asd].copy()
    asd_df = asd_df[asd_df[group2_col].isin([0, 2])].copy()
    y_asd  = (asd_df[group2_col] == 2).astype(int).to_numpy()

    print(f"\nASD training participants (Group2 extremes only): {len(y_asd)}")
    print(f"Top tertile (label=1): {y_asd.sum()} | "
          f"Bottom tertile (label=0): {(y_asd == 0).sum()}")

    # ─────────────────────────────────────────────
    # Feature matrices — ASD
    # ─────────────────────────────────────────────
    feature_matrices_asd = {}
    for col in feature_cols:
        try:
            X = np.vstack(asd_df[col].to_numpy())
        except Exception as e:
            print(f"Error stacking {col} for ASD set: {e}")
            continue
        feature_matrices_asd[col.replace(" features", "")] = X

    if control_covariates:
        print("\nAdding covariates: Age + Sex to ASD models")
        cov_asd = np.hstack([
            asd_df["Age_years"].to_numpy()[:, None],
            asd_df["Sex"].map({"M": 0, "F": 1}).to_numpy()[:, None]
        ])
    else:
        cov_asd = None

    combined_asd_parts = list(feature_matrices_asd.values())
    if control_covariates:
        combined_asd_parts.append(cov_asd)
    X_asd_combined = np.hstack(combined_asd_parts)

    # ─────────────────────────────────────────────
    # PMS vs TD test set
    # ─────────────────────────────────────────────
    test_df = df[df["Group"].isin([group_pms, group_td])].copy()
    print(f"\nPMS/TD test participants: {len(test_df)}")
    y_test  = (test_df["Group"] == group_pms).astype(int).to_numpy()

    feature_matrices_test = {}
    for col in feature_cols:
        try:
            X = np.vstack(test_df[col].to_numpy())
        except Exception as e:
            print(f"Error stacking {col} for PMS/TD set: {e}")
            continue
        feature_matrices_test[col.replace(" features", "")] = X

    if control_covariates:
        cov_test = np.hstack([
            test_df["Age_years"].to_numpy()[:, None],
            test_df["Sex"].map({"M": 0, "F": 1}).to_numpy()[:, None]
        ])
    else:
        cov_test = None

    combined_test_parts = [feature_matrices_test[n]
                           for n in feature_matrices_asd if n in feature_matrices_test]
    if control_covariates:
        combined_test_parts.append(cov_test)
    X_test_combined = np.hstack(combined_test_parts)

    # ─────────────────────────────────────────────
    # Per-feature models
    # ─────────────────────────────────────────────
    results       = {}
    fitted_models = {}

    sep  = "=" * 62
    thin = "-" * 62

    print(f"\n{sep}")
    print("  MODEL PERFORMANCE — ASD Group2 (0 vs 2) + PMS vs TD Test")
    print(sep)

    for name, X_asd in feature_matrices_asd.items():

        X_asd_model  = np.hstack([X_asd,  cov_asd])  if control_covariates else X_asd
        X_test_model = np.hstack([feature_matrices_test[name], cov_test]) \
                       if control_covariates else feature_matrices_test[name]

        if X_asd_model.shape[1] != X_test_model.shape[1]:
            print(f"Shape mismatch for {name}, skipping. "
                  f"ASD: {X_asd_model.shape}, Test: {X_test_model.shape}")
            continue

        print(f"\n  Feature: {name}")
        print(thin)

        # LOO on ASD
        clf = make_clf()
        y_true_asd, y_scores_asd, auc_asd = loo_predictions(X_asd_model, y_asd, clf)
        print(f"  ASD LOO AUROC  (Group2 0 vs 2) : {auc_asd:.3f}")

        # Fit final model on all ASD
        clf_full = make_clf()
        clf_full.fit(X_asd_model, y_asd)
        fitted_models[name] = clf_full

        # Observed test AUROC
        y_scores_test = clf_full.predict_proba(X_test_model)[:, 1]
        auc_test = roc_auc_score(y_test, y_scores_test)
        print(f"  PMS vs TD AUROC (observed)     : {auc_test:.3f}")

        # Label-shuffle null distribution
        null_aurocs, null_mean, null_std, p_val = shuffle_null(
            clf_full, X_test_model, y_test, n_label_shuffles, rng
        )
        print(f"  Null AUROC ({n_label_shuffles} shuffles)       : "
              f"{null_mean:.3f} ± {null_std:.3f}")
        print(f"  Permutation p-value            : {p_val:.3f}"
              + (" *" if p_val < 0.05 else ""))

        results[name] = {
            "ASD": {
                "y_true": y_true_asd,
                "y_scores": y_scores_asd,
                "auroc": auc_asd,
            },
            "PMSvsTD": {
                "y_true": y_test,
                "y_scores": y_scores_test,
                "auroc": auc_test,
                "shuffle_null": {
                    "aurocs": null_aurocs,
                    "mean":   null_mean,
                    "std":    null_std,
                    "p_value": p_val,
                    "n_shuffles": n_label_shuffles,
                }
            }
        }

    # ─────────────────────────────────────────────
    # Covariates-only model
    # ─────────────────────────────────────────────
    if control_covariates:
        print(f"\n{thin}")
        print("  Covariates-Only Model (Age + Sex)")
        print(thin)

        clf = make_clf()
        y_true_asd, y_scores_asd, auc_asd = loo_predictions(cov_asd, y_asd, clf)
        print(f"  ASD LOO AUROC  (Group2 0 vs 2) : {auc_asd:.3f}")

        clf_full = make_clf()
        clf_full.fit(cov_asd, y_asd)
        fitted_models["CovariatesOnly"] = clf_full

        y_scores_test = clf_full.predict_proba(cov_test)[:, 1]
        auc_test = roc_auc_score(y_test, y_scores_test)
        print(f"  PMS vs TD AUROC (observed)     : {auc_test:.3f}")

        null_aurocs, null_mean, null_std, p_val = shuffle_null(
            clf_full, cov_test, y_test, n_label_shuffles, rng
        )
        print(f"  Null AUROC ({n_label_shuffles} shuffles)       : "
              f"{null_mean:.3f} ± {null_std:.3f}")
        print(f"  Permutation p-value            : {p_val:.3f}"
              + (" *" if p_val < 0.05 else ""))

        results["CovariatesOnly"] = {
            "ASD": {
                "y_true": y_true_asd,
                "y_scores": y_scores_asd,
                "auroc": auc_asd,
            },
            "PMSvsTD": {
                "y_true": y_test,
                "y_scores": y_scores_test,
                "auroc": auc_test,
                "shuffle_null": {
                    "aurocs": null_aurocs,
                    "mean":   null_mean,
                    "std":    null_std,
                    "p_value": p_val,
                    "n_shuffles": n_label_shuffles,
                }
            }
        }

    # ─────────────────────────────────────────────
    # Combined feature model
    # ─────────────────────────────────────────────
    print(f"\n{thin}")
    print("  Combined Feature Model")
    print(thin)

    clf = make_clf()
    y_true_asd, y_scores_asd, auc_asd = loo_predictions(X_asd_combined, y_asd, clf)
    print(f"  ASD LOO AUROC  (Group2 0 vs 2) : {auc_asd:.3f}")

    clf_full = make_clf()
    clf_full.fit(X_asd_combined, y_asd)
    fitted_models["Combined"] = clf_full

    y_scores_test = clf_full.predict_proba(X_test_combined)[:, 1]
    auc_test = roc_auc_score(y_test, y_scores_test)
    print(f"  PMS vs TD AUROC (observed)     : {auc_test:.3f}")

    null_aurocs, null_mean, null_std, p_val = shuffle_null(
        clf_full, X_test_combined, y_test, n_label_shuffles, rng
    )
    print(f"  Null AUROC ({n_label_shuffles} shuffles)       : "
          f"{null_mean:.3f} ± {null_std:.3f}")
    print(f"  Permutation p-value            : {p_val:.3f}"
          + (" *" if p_val < 0.05 else ""))

    print(f"\n{sep}\n")

    results["Combined"] = {
        "ASD": {
            "y_true": y_true_asd,
            "y_scores": y_scores_asd,
            "auroc": auc_asd,
        },
        "PMSvsTD": {
            "y_true": y_test,
            "y_scores": y_scores_test,
            "auroc": auc_test,
            "shuffle_null": {
                "aurocs": null_aurocs,
                "mean":   null_mean,
                "std":    null_std,
                "p_value": p_val,
                "n_shuffles": n_label_shuffles,
            }
        }
    }

    return results, fitted_models

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


def plot_roc_curves(
        results,
        filename,
        subset="ASD",   # "ASD" → Group2 0vs2 training ROC, "PMSvsTD" → generalization ROC
        title="ROC Curves for Classifier Models"
    ):
    """
    Plot ROC curves from the nested results dictionary produced by
    classifier_group2_from_asd().

    Parameters
    ----------
    results : dict
        Dictionary such that:
            results[model]["ASD"]     = {y_true, y_scores, auroc}
            results[model]["PMSvsTD"] = {y_true, y_scores, auroc}

    subset : str
        Either "ASD" or "PMSvsTD" to choose which ROC curves to plot.

    title : str
        Title of the plot.
    """

    # Validate subset
    if subset not in ["ASD", "PMSvsTD"]:
        raise ValueError("subset must be 'ASD' or 'PMSvsTD'.")

    plt.figure(figsize=(7, 6))

    # Detect covariates-only model
    has_cov_only = "CovariatesOnly" in results

    colors = plt.cm.tab10.colors

    # --------------------------------------------------------
    # 1. Identify best-performing model for chosen subset
    # --------------------------------------------------------
    aucs = {}
    for model_name, data in results.items():
        if subset in data and data[subset].get("auroc") is not None:
            aucs[model_name] = data[subset]["auroc"]

    if len(aucs) == 0:
        raise RuntimeError(f"No AUROC values found for subset '{subset}'.")

    best_model = max(aucs, key=aucs.get)

    # --------------------------------------------------------
    # 2. Plot each ROC
    # --------------------------------------------------------
    for i, (model_name, data) in enumerate(results.items()):

        if subset not in data:
            continue

        y_true = data[subset].get("y_true")
        y_scores = data[subset].get("y_scores")

        if y_true is None or y_scores is None:
            continue

        # Compute ROC curve
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)

        # Highlight best model
        alpha = 1.0 if model_name == best_model else 0.55
        linewidth = 2.5 if model_name == best_model else 1.2

        # --------------------------------------------------------
        # Legend formatting rules
        # --------------------------------------------------------
        if has_cov_only:
            if model_name == "CovariatesOnly":
                label = f"Age + Sex only (AUC = {roc_auc:.3f})"
            elif model_name == "Combined":
                label = f"{model_name} (AUC = {roc_auc:.3f})"
            else:
                label = f"{model_name} + Age + Sex (AUC = {roc_auc:.3f})"
        else:
            label = f"{model_name} (AUC = {roc_auc:.3f})"

        plt.plot(
            fpr,
            tpr,
            color=colors[i % len(colors)],
            linewidth=linewidth,
            alpha=alpha,
            label=label
        )

    # --------------------------------------------------------
    # 3. Chance line
    # --------------------------------------------------------
    plt.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.4)

    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(filename, format="svg", bbox_inches='tight')
    plt.show()


In [ ]:
results_full, models_full = classifier_group2_from_asd(
    merged_df,
    group_asd="ASD",
    group_pms="PMS",
    group_td="TD",
    group2_col="Group2 FULL",
    control_covariates=True,
    random_state=42
)


In [ ]:
plot_roc_curves(results_full, subset="PMSvsTD",
                title="ROC Curves for ASD Model Predicting SAI (Top vs Bottom Tertile) \n Tested on Unseen PMS vs TD (w/ Age + Sex Covariates)",
                filename = 'figures/2b.svg')

In [ ]:
results_itpc, models_itpc = classifier_group2_from_asd(
    merged_df,
    group_asd="ASD",
    group_pms="PMS",
    group_td="TD",
    group2_col="Group2 ITPC",
    control_covariates=False,
    random_state=42
)


In [ ]:
plot_roc_curves(results_itpc, subset="PMSvsTD",
                title="ROC Curves for ASD Model Predicting SAI (Top vs Bottom Tertile) \n Tested on Unseen PMS vs TD (w/o Age + Sex Covariates)",
                filename='figures/S1b.svg')